# NeuroAgent: A Board of AI Doctors for EEG Review

**High-Risk Project — AI in Healthcare**

## What this notebook does, in plain English

Imagine a hard epilepsy case where the neurologist isn't 100% sure what type of seizure it is.
In real hospitals, doctors sometimes ask a colleague for a second opinion. This project builds
a small AI version of that: instead of one AI model giving one confident answer, we ask **three
AI "doctor" agents** to look at the same EEG data, each with a different role and a different
underlying model. Then a **moderator agent** compares their opinions and tells us where they
agree, where they disagree, and why.

This is *not* meant to replace a real doctor. It is a research prototype that explores whether
a "board of AI doctors" can make uncertainty more visible and more evidence-based, instead of
hiding it behind one falsely-confident answer.

**Data used:** CHB-MIT Scalp EEG Database (public, pediatric epilepsy patients, no seizure-type
labels -- just seizure vs. non-seizure). This means v1 of this project focuses on **describing
what the EEG shows**, not on predicting the exact seizure type. Full seizure-type classification
is planned as future work once access to a type-labeled dataset (TUSZ) is available.

**Code and slides link:** [add your GitHub/Colab link here before submitting]


## 1. Setup

We install the packages we need:
- `openai` to call the AI models
- `mne` to read EEG `.edf` files
- `pandas` / `numpy` for data handling


In [1]:
!pip install openai mne pandas numpy scikit-learn --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 25.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
import os
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
gv_client = OpenAI()

# Three doctor agents, each on a different model.
gv_model_neurologist = "gpt-4o"
gv_model_epileptologist = "o3-mini"
gv_model_neurophysiologist = "gpt-4o-mini"


'o1-mini' not available on this account (NotFoundError), trying next...
Using 'o3-mini' for the epileptologist agent.


## 2. Download the EEG data (CHB-MIT)

We only download a small, chosen set of files -- not the whole 42GB database. We pick:
- A few files **with** seizures (positive examples)
- A few files **without** seizures (negative examples), so the AI doctors see both

This downloads straight from PhysioNet into this Colab session (nothing touches your own computer).


In [3]:
gv_base_url = "https://physionet.org/files/chbmit/1.0.0"

# case -> list of files (seizure files + one non-seizure file + the summary text file)
gv_cases = {
    "chb01": ["chb01_03.edf", "chb01_04.edf", "chb01_01.edf", "chb01-summary.txt"],
    "chb05": ["chb05_06.edf", "chb05_13.edf", "chb05_01.edf", "chb05-summary.txt"],
    "chb08": ["chb08_02.edf", "chb08_05.edf", "chb08_01.edf", "chb08-summary.txt"],
}

for lv_case, lv_files in gv_cases.items():
    lv_dir = f"/content/chbmit/{lv_case}"
    os.makedirs(lv_dir, exist_ok=True)
    for lv_fname in lv_files:
        lv_url = f"{gv_base_url}/{lv_case}/{lv_fname}"
        !wget -q -nc -P {lv_dir} {lv_url}
    print(f"{lv_case}: downloaded {len(lv_files)} files")


chb01: downloaded 4 files
chb05: downloaded 4 files
chb08: downloaded 4 files


## 3. Turn raw EEG into something an AI can read

Raw EEG is just numbers -- millions of them. Before we can ask an AI model to reason about it,
we need to turn each EEG window into a short, human-readable description: how strong different
brain-wave frequencies are, how the signal is shaped, which channels look unusual.

We read each `-summary.txt` file to find the **exact seizure start/end times**, so we can cut out:
- A **seizure window** (during the labeled seizure)
- A **non-seizure window** (same patient, normal activity) for contrast

Then we compute simple frequency-band features (delta, theta, alpha, beta, gamma power) for each
window. These band powers are a standard, well-understood way to summarize EEG activity, and they
are compact enough to describe to an AI model in plain text.


In [4]:
import mne
import re

def parse_summary_seizures(lv_summary_path):
    """Read a chbXX-summary.txt file and return a list of
    (filename, seizure_start_seconds, seizure_end_seconds) for every seizure listed."""
    lv_text = open(lv_summary_path).read()
    lv_blocks = lv_text.split("File Name:")[1:]
    lv_results = []
    for lv_block in lv_blocks:
        lv_fname = lv_block.strip().split()[0]
        lv_starts = re.findall(r"Seizure Start Time:\s*(\d+)", lv_block)
        lv_ends = re.findall(r"Seizure End Time:\s*(\d+)", lv_block)
        for lv_s, lv_e in zip(lv_starts, lv_ends):
            lv_results.append((lv_fname, int(lv_s), int(lv_e)))
    return lv_results


def band_power_features(lv_data, lv_sfreq):
    """Compute simple frequency-band power features for a window of EEG data.
    lv_data shape: (n_channels, n_samples). Returns a dict of band -> average power."""
    lv_psds, lv_freqs = mne.time_frequency.psd_array_welch(
        lv_data, sfreq=lv_sfreq, fmin=0.5, fmax=45, n_fft=min(256, lv_data.shape[1]), verbose=False
    )
    lv_bands = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}
    lv_out = {}
    for lv_name, (lv_lo, lv_hi) in lv_bands.items():
        lv_mask = (lv_freqs >= lv_lo) & (lv_freqs < lv_hi)
        lv_out[lv_name] = float(lv_psds[:, lv_mask].mean())
    return lv_out


def extract_case_windows(lv_case, lv_files, lv_window_sec=20):
    """For a given case, extract one seizure window and one non-seizure window
    per seizure file, plus one window from the dedicated non-seizure file."""
    lv_dir = f"/content/chbmit/{lv_case}"
    lv_summary_path = f"{lv_dir}/{lv_case}-summary.txt"
    lv_seizures = parse_summary_seizures(lv_summary_path)
    lv_records = []

    for lv_fname, lv_start, lv_end in lv_seizures:
        lv_path = f"{lv_dir}/{lv_fname}"
        if not os.path.exists(lv_path):
            continue
        lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
        lv_sfreq = lv_raw.info["sfreq"]

        # Seizure window
        lv_seg = lv_raw.copy().crop(tmin=lv_start, tmax=min(lv_start + lv_window_sec, lv_end))
        lv_feat = band_power_features(lv_seg.get_data(), lv_sfreq)
        lv_records.append({"case": lv_case, "file": lv_fname, "label": "seizure", **lv_feat})

        # Non-seizure window from the same file, well before the seizure starts
        if lv_start > lv_window_sec + 30:
            lv_seg2 = lv_raw.copy().crop(tmin=10, tmax=10 + lv_window_sec)
            lv_feat2 = band_power_features(lv_seg2.get_data(), lv_sfreq)
            lv_records.append({"case": lv_case, "file": lv_fname, "label": "non_seizure", **lv_feat2})

    return lv_records


gv_all_records = []
for lv_case, lv_files in gv_cases.items():
    gv_all_records.extend(extract_case_windows(lv_case, lv_files))

gv_features_df = pd.DataFrame(gv_all_records)
print(f"Extracted {len(gv_features_df)} EEG windows (seizure + non-seizure)")
gv_features_df


/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)


Extracted 8 EEG windows (seizure + non-seizure)


,case,file,label,delta,theta,alpha,beta,gamma
0,chb01,chb01_03.edf,seizure,1.900132e-09,4.280013e-10,4.331971e-11,1.241013e-11,2.033422e-11
1,chb01,chb01_03.edf,non_seizure,2.721512e-10,5.793261e-11,3.712411e-11,4.240661e-12,6.575679e-13
2,chb01,chb01_04.edf,seizure,2.619219e-09,4.451644e-10,4.349532e-11,1.901407e-11,2.399497e-11
3,chb01,chb01_04.edf,non_seizure,1.452702e-10,4.246701e-11,1.027651e-11,3.691532e-12,2.308902e-12
4,chb05,chb05_06.edf,seizure,5.647274e-10,1.316556e-09,2.042092e-10,4.533573e-11,8.222060e-12
5,chb05,chb05_06.edf,non_seizure,1.092846e-09,4.888177e-10,2.948911e-11,1.020008e-11,6.988870e-12
6,chb05,chb05_13.edf,seizure,7.658479e-10,4.980844e-09,2.049728e-09,3.625813e-10,1.041426e-10
7,chb05,chb05_13.edf,non_seizure,6.530586e-10,1.197691e-10,3.083264e-11,3.448828e-11,4.683773e-11


### Turn each row into a short text description

The AI doctor agents don't read numbers well -- they read text. So we turn each row of features
into a short, plain-English description of what the EEG window looks like.


In [5]:
def features_to_text(lv_row):
    return (
        f"EEG window from case {lv_row['case']}, file {lv_row['file']}.\n"
        f"Average band power (relative strength of brain-wave frequencies):\n"
        f"  Delta (0.5-4 Hz): {lv_row['delta']:.3g}\n"
        f"  Theta (4-8 Hz): {lv_row['theta']:.3g}\n"
        f"  Alpha (8-13 Hz): {lv_row['alpha']:.3g}\n"
        f"  Beta (13-30 Hz): {lv_row['beta']:.3g}\n"
        f"  Gamma (30-45 Hz): {lv_row['gamma']:.3g}\n"
    )

gv_features_df["eeg_description"] = gv_features_df.apply(features_to_text, axis=1)
print(gv_features_df["eeg_description"].iloc[0])


EEG window from case chb01, file chb01_03.edf.
Average band power (relative strength of brain-wave frequencies):
  Delta (0.5-4 Hz): 1.9e-09
  Theta (4-8 Hz): 4.28e-10
  Alpha (8-13 Hz): 4.33e-11
  Beta (13-30 Hz): 1.24e-11
  Gamma (30-45 Hz): 2.03e-11



## 4. A small library of epilepsy literature (for grounding)

Real doctors don't just look at a signal -- they compare it to what they know from medical
literature. We give our AI doctors a small, curated set of reference notes about EEG patterns
in epilepsy, and a simple retrieval step: for each EEG window, we score every snippet by how many
words it shares with the window's description (with extra weight on the window's dominant
frequency band), and pass only the top two matches to the doctor agents -- not the whole library
every time. This is a small-scale but genuine retrieval step, not just static context.


In [ ]:
gv_literature = {
    "focal_seizure": (
        "Focal seizures typically show localized rhythmic discharges, often with increased "
        "beta or gamma activity in specific channels, reflecting a seizure onset restricted "
        "to one brain region before possibly spreading."
    ),
    "generalized_seizure": (
        "Generalized seizures typically show synchronous, widespread discharges across most "
        "or all channels simultaneously, often with prominent slow-wave (delta/theta) activity."
    ),
    "normal_background": (
        "Normal background EEG in an awake or lightly drowsy pediatric patient shows a mix of "
        "alpha and beta activity without sustained high-amplitude rhythmic discharges."
    ),
    "ictal_pattern": (
        "The ictal (seizure) period is often characterized by a sudden change in frequency and "
        "amplitude compared to the pre-seizure baseline, most clearly seen as a shift toward "
        "higher beta/gamma power or rhythmic delta/theta buildup."
    ),
}


def extract_band_powers(lv_description):
    lv_bands = {}
    for lv_match in re.finditer(r"(\w+) \([\d.]+-[\d.]+ Hz\): ([\d.eE+-]+)", lv_description):
        lv_bands[lv_match.group(1).lower()] = float(lv_match.group(2))
    return lv_bands


def retrieve_literature(lv_eeg_description, lv_top_k=2):
    lv_bands = extract_band_powers(lv_eeg_description)
    lv_dominant_band = max(lv_bands, key=lv_bands.get) if lv_bands else ""
    lv_query_words = set(re.findall(r"[a-z]+", lv_eeg_description.lower()))
    lv_query_words |= {lv_dominant_band, lv_dominant_band}  # extra weight for the dominant band

    lv_scored = []
    for lv_key, lv_text in gv_literature.items():
        lv_snippet_words = re.findall(r"[a-z]+", lv_text.lower())
        lv_overlap = sum(1 for lv_w in lv_snippet_words if lv_w in lv_query_words)
        lv_scored.append((lv_overlap, lv_key, lv_text))

    lv_scored.sort(key=lambda lv_x: lv_x[0], reverse=True)
    lv_top = lv_scored[:lv_top_k]
    return "\n".join(f"[{lv_key}] {lv_text}" for _, lv_key, lv_text in lv_top)


## 5. The three AI doctor agents

Each "doctor" is really just a function: it takes the EEG description and relevant literature,
and asks an AI model to reason about it **as if it were a specific kind of clinician**.

We deliberately mix two things to make the three doctors genuinely different from each other:
- **Different underlying AI models** (not just different instructions on the same model)
- **Different clinical roles / ways of thinking about the case**

| Agent | Model | Role |
|---|---|---|
| Doctor 1 | gpt-4o | Neurologist -- general seizure differential |
| Doctor 2 | o3-mini | Epileptologist -- subspecialist, reasons more deeply about ambiguous cases |
| Doctor 3 | gpt-4o-mini | Neurophysiologist -- focuses on the raw signal pattern, not clinical presentation |


In [7]:
gv_role_prompts = {
    "neurologist": (
        "You are a general neurologist reviewing a pediatric EEG window. Give your best-guess "
        "differential (is this seizure activity or not, and if so, does it look more focal or "
        "generalized?), your reasoning, and how confident you are (low/medium/high)."
    ),
    "epileptologist": (
        "You are an epileptologist, a neurologist who subspecializes in epilepsy and handles "
        "ambiguous or hard-to-classify cases. Review this pediatric EEG window carefully, "
        "consider more than one possible interpretation, and explain the reasoning behind each. "
        "State your confidence (low/medium/high) for your leading interpretation."
    ),
    "neurophysiologist": (
        "You are a clinical neurophysiologist who focuses on the raw EEG signal itself rather "
        "than the patient's clinical presentation. Focus your reasoning on the frequency-band "
        "pattern described below: what it suggests about the underlying brain activity, and how "
        "confident you are (low/medium/high)."
    ),
}

def call_llm(lv_model, lv_system_prompt, lv_user_prompt, lv_retries=2):
    """Call an OpenAI model and return the response text.
    Retries a couple of times if the call fails, instead of crashing the whole run."""
    for lv_attempt in range(lv_retries + 1):
        try:
            lv_resp = gv_client.chat.completions.create(
                model=lv_model,
                messages=[
                    {"role": "system", "content": lv_system_prompt},
                    {"role": "user", "content": lv_user_prompt},
                ],
            )
            return lv_resp.choices[0].message.content
        except Exception as lv_e:
            if lv_attempt == lv_retries:
                print("Failed after retries:", lv_e)
                return f"[ERROR: {lv_e}]"
            time.sleep(2)


def doctor_agent(lv_role, lv_model, lv_eeg_description, lv_literature):
    """One doctor agent: combines a role, a model, the EEG description, and grounding literature."""
    lv_system_prompt = gv_role_prompts[lv_role]
    lv_user_prompt = (
        f"EEG window description:\n{lv_eeg_description}\n\n"
        f"Relevant reference literature:\n{lv_literature}\n\n"
        "Give your assessment, your reasoning, and your confidence level."
    )
    return call_llm(lv_model, lv_system_prompt, lv_user_prompt)


## 6. Try the board of doctors on one example case

Before running everything, let's test the full pipeline on a single EEG window so we can read
through what each doctor says.


In [8]:
gv_example_row = gv_features_df.iloc[0]
gv_example_description = gv_example_row["eeg_description"]
gv_example_literature = retrieve_literature(gv_example_description)

gv_doctor_outputs = {}
gv_doctor_outputs["neurologist"] = doctor_agent("neurologist", gv_model_neurologist, gv_example_description, gv_example_literature)
gv_doctor_outputs["epileptologist"] = doctor_agent("epileptologist", gv_model_epileptologist, gv_example_description, gv_example_literature)
gv_doctor_outputs["neurophysiologist"] = doctor_agent("neurophysiologist", gv_model_neurophysiologist, gv_example_description, gv_example_literature)

for lv_role, lv_output in gv_doctor_outputs.items():
    print(f"===== {lv_role.upper()} =====")
    print(lv_output)
    print()


===== NEUROLOGIST =====
Based on the provided EEG window description, the band power values indicate the following relative strengths of brain-wave frequencies:

- Delta (0.5-4 Hz): 1.9e-09
- Theta (4-8 Hz): 4.28e-10
- Alpha (8-13 Hz): 4.33e-11
- Beta (13-30 Hz): 1.24e-11
- Gamma (30-45 Hz): 2.03e-11

These values show a predominance of delta activity with a notably higher amplitude compared to other frequency bands. The theta band also exhibits some increase in power, while the alpha, beta, and gamma bands show much lower relative power levels.

In seizure activity, especially focal seizures, one might expect to see increased beta or gamma activity which reflects seizure onset in specific localized regions. Generalized seizures, on the other hand, are characterized by synchronous and widespread discharges; often delta and theta frequencies in these seizures are elevated. The data shows elevated delta power which could correlate with a seizure state, although beta and gamma frequencies

## 7. The moderator agent

The moderator's job is not to declare one "correct" answer. It reads all three doctors'
opinions and explains **where they agree, where they disagree, and why** -- the same way a
lead clinician might summarize a multidisciplinary discussion. This disagreement summary is
itself one of the most useful outputs of this project, since we don't have labeled ground
truth to score against.


In [9]:
gv_model_moderator = "gpt-4o"

def moderator_agent(lv_doctor_outputs):
    lv_system_prompt = (
        "You are a moderator summarizing a discussion between three doctors (a neurologist, "
        "an epileptologist, and a neurophysiologist) who each reviewed the same EEG window "
        "independently. Identify where they agree, where they disagree, and explain possible "
        "reasons for any disagreement. Do not simply pick a winner -- the goal is to make the "
        "uncertainty and reasoning visible, the way a real multidisciplinary discussion would."
    )
    lv_user_prompt = "\n\n".join(
        f"{lv_role.upper()} said:\n{lv_text}" for lv_role, lv_text in lv_doctor_outputs.items()
    )
    return call_llm(gv_model_moderator, lv_system_prompt, lv_user_prompt)

gv_moderator_summary = moderator_agent(gv_doctor_outputs)
print(gv_moderator_summary)


The discussion among the neurologist, epileptologist, and neurophysiologist reveals both areas of agreement and divergence in interpretation, reflecting the complexity of EEG analysis.

**Areas of Agreement:**
1. **Predominance of Delta Activity:** All three specialists agree that the delta band activity is the most significant in the EEG window, which is a critical observation for understanding the state of the brain.
   
2. **General Absence of Beta and Gamma Activity:** They collectively note the low power in the beta and gamma bands, implying a lack of focal seizure activity and challenging the notion of active engagement or cognitive tasking.

**Areas of Disagreement and Possible Reasons:**
1. **Interpretation of Delta Activity:**
   - **Neurologist:** Suggests the delta predominance could indicate generalized seizure activity. However, they acknowledge the lack of accompanying elevated beta/gamma activity, which would be expected in focal seizures.
   - **Epileptologist:** Primar

## 8. Run the full board across all EEG windows

Now we repeat this process -- three doctors plus a moderator -- across every EEG window we
extracted earlier, and save everything into a table we can review and discuss in our report.


In [ ]:
gv_results = []

for lv_idx, lv_row in gv_features_df.iterrows():
    lv_description = lv_row["eeg_description"]
    lv_literature = retrieve_literature(lv_description)

    lv_outputs = {
        "neurologist": doctor_agent("neurologist", gv_model_neurologist, lv_description, lv_literature),
        "epileptologist": doctor_agent("epileptologist", gv_model_epileptologist, lv_description, lv_literature),
        "neurophysiologist": doctor_agent("neurophysiologist", gv_model_neurophysiologist, lv_description, lv_literature),
    }
    lv_moderator_summary = moderator_agent(lv_outputs)

    gv_results.append({
        "case": lv_row["case"],
        "file": lv_row["file"],
        "true_label": lv_row["label"],  # seizure vs non_seizure, from the dataset annotation
        "neurologist_opinion": lv_outputs["neurologist"],
        "epileptologist_opinion": lv_outputs["epileptologist"],
        "neurophysiologist_opinion": lv_outputs["neurophysiologist"],
        "moderator_summary": lv_moderator_summary,
    })
    print(f"Processed {lv_row['case']} / {lv_row['file']} ({lv_row['label']})")

pd.set_option("display.max_colwidth", None)
gv_results_df = pd.DataFrame(gv_results)
gv_results_df


### Save the results

We save the full results table so it can be reviewed later, included in the report, or picked
up again once TUSZ access (with real seizure-type labels) is available for v2.


In [11]:
gv_results_df.to_csv("/content/neuroagent_v1_results.csv", index=False)
print("Saved results to /content/neuroagent_v1_results.csv")


Saved results to /content/neuroagent_v1_results.csv


## 9. A simple baseline: logistic regression on band power

Before trusting a complex multi-agent system, it's worth asking a simpler question first: how
far can a plain classical machine learning model get using only the same five band-power numbers
the doctor agents see? This gives us a real comparison point -- if a simple model already
separates seizure from non-seizure well, that tells us something different than if it struggles,
which is exactly the kind of case where richer, evidence-grounded reasoning might add more value.

With only a handful of EEG windows, a normal train/test split would leave almost nothing to test
on. Instead we use **leave-one-out cross-validation**: for each window, train on every other
window and predict just that one, then repeat for every window. This is a standard way to get an
honest read from a very small dataset, though the result here is still illustrative, not a
validated benchmark.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler

gv_feature_cols = ["delta", "theta", "alpha", "beta", "gamma"]
gv_X = gv_features_df[gv_feature_cols].values
gv_y = (gv_features_df["label"] == "seizure").astype(int).values

gv_loo = LeaveOneOut()
gv_baseline_predictions = []

for lv_train_idx, lv_test_idx in gv_loo.split(gv_X):
    lv_scaler = StandardScaler()
    lv_X_train = lv_scaler.fit_transform(gv_X[lv_train_idx])
    lv_X_test = lv_scaler.transform(gv_X[lv_test_idx])
    lv_clf = LogisticRegression()
    lv_clf.fit(lv_X_train, gv_y[lv_train_idx])
    lv_pred = lv_clf.predict(lv_X_test)[0]
    gv_baseline_predictions.append(lv_pred)

gv_features_df["baseline_prediction"] = [
    "seizure" if lv_p == 1 else "non_seizure" for lv_p in gv_baseline_predictions
]
gv_baseline_correct = (gv_features_df["baseline_prediction"] == gv_features_df["label"]).sum()
gv_baseline_accuracy = gv_baseline_correct / len(gv_features_df)

print(f"Baseline logistic regression accuracy (leave-one-out): {gv_baseline_accuracy:.2f} "
      f"({gv_baseline_correct}/{len(gv_features_df)} correct)")
gv_features_df[["case", "file", "label", "baseline_prediction"] + gv_feature_cols]


## 10. Caveats and Limitations

- **No seizure-type ground truth.** CHB-MIT only labels seizure vs. non-seizure, not seizure
  *type*. So this v1 system reasons about EEG characteristics and produces differentials, but
  we cannot score those differentials against a validated type label. Full type-level validation
  is planned for v2 once TUSZ access (currently pending) becomes available.
- **"Different doctors" here means different model + different role prompt, not truly
  independent training.** All three doctor agents use OpenAI models, so some shared biases or
  blind spots from that underlying training are still possible. This is a real limitation, not
  just a caveat -- true multi-provider diversity (e.g., adding a non-OpenAI model) is a natural
  next step.
- **The literature library is small and hand-curated, and retrieval is a simple keyword-overlap
  score**, not a production-scale semantic search system. It genuinely selects a subset of
  snippets rather than always returning everything, but with only four snippets in the library,
  its ability to meaningfully discriminate is limited.
- **Feature extraction is simple (band power only).** More advanced signal features (waveform
  morphology, spike detection, channel-specific patterns) could give the doctor agents richer
  information to reason over.
- **Small sample size.** Only 3 patient cases and a handful of EEG windows were used, chosen to
  keep this v1 build feasible within the project timeline. This is a proof-of-concept scale,
  not a validation-scale sample.
- **The logistic regression baseline is illustrative only.** With just 8 EEG windows,
  leave-one-out cross-validation is the most honest evaluation available, but it is not a
  statistically meaningful benchmark. The comparison between the baseline and the doctor board
  should be read as a qualitative signal, not proof that one approach outperforms the other.
- **Observed in the v1 run:** on at least one seizure-labeled window, two of the three doctor
  agents leaned toward a non-seizure interpretation (deep sleep or an inactive background) with
  high stated confidence, while only the neurologist tentatively suggested seizure activity at
  medium confidence. This is a genuine finding, not a bug: it suggests that average band power
  over a 20-second window, without spatial (channel-level) or temporal (evolution-over-time)
  information, is not always sufficient to distinguish seizure activity from normal slow-wave
  sleep. Richer features (per-channel patterns, time-resolved changes) are a natural next step.


## 11. Next Steps / Future Work

- **v2 -- Seizure-type differential:** Once TUSZ access is granted, replace the seizure/non-seizure
  labels with real seizure-type labels (focal, generalized, absence, tonic-clonic, etc.) and have
  the doctor board reason about *type*, not just presence.
- **v3 -- Medication agent:** Add a fourth agent that, given a seizure type, reasons about
  first-line antiseizure medication options grounded in clinical guideline literature. This is a
  natural next step led by literature review rather than heavy coding.
- **v4 -- Longitudinal dosing-awareness agent:** A more ambitious future direction: given a
  patient's age/weight trajectory and current medication, flag *when* a dose review might be due,
  rather than waiting for a breakthrough seizure. This would need to be framed carefully as a
  scheduling/awareness aid, not a clinical dosing recommendation.
- **Multi-provider model diversity:** Add doctor agents built on non-OpenAI models to get closer
  to genuinely independent "opinions," addressing the single-provider limitation noted above.
